# Page Rank Algorithm

PageRank 是一种用于评估有向图中节点重要性的方法。**核心功能**：衡量网络中节点的重要性。

常用于**网页排序**，**社交分析** 等。


1. **核心思想**：
   - **投票机制**：一个节点的重要性由指向它的其他节点决定，且这些节点自身的重要性越高，其“投票”权重越大。
   - **模拟随机游走**：假设用户在图中随机跳转（点击链接或随机访问），经过长时间游走，用户访问每个网页的概率收敛到稳定值，即该网页的**PageRank值**。值越高，网页越重要。

2. 算法介绍

- **转移矩阵`M`**：若页面`j`有`k`个出链，则每个出链的转移概率为`1/k`（即`M`的每列和为1）。
- **平稳分布**：通过方程`MR=R`求解，`R`即为各节点的PageRank值。

针对特殊情况的优化：

- **悬挂节点**（无出链的节点）：用户无处可去。
- **非强连通图**：存在无法相互访问的节点群。
- **周期性**：某些节点间形成循环，导致概率无法收敛。

**解决方案**：引入**阻尼因子`d`**，将转移分为两部分：

- 以概率`d`按原链接跳转（解决周期性）。
- 以概率`1-d`随机跳转到任意节点（解决悬挂节点和非连通性）。

更新公式：
$$R = d \cdot M R + \frac{1-d}{n} \cdot \textbf{1}


In [2]:
import networkx as nx 
import numpy as np
barabasi_graph = nx.barabasi_albert_graph(60,41) 
matrix = nx.to_numpy_array(barabasi_graph, dtype = np.int16)
print(matrix)
pr = nx.pagerank(barabasi_graph,0.85) 
print(np.array( [v for k, v in pr.items()]))

[[0 1 1 ... 1 1 1]
 [1 0 0 ... 1 1 1]
 [1 0 0 ... 0 0 0]
 ...
 [1 1 0 ... 0 0 1]
 [1 1 0 ... 0 0 1]
 [1 1 0 ... 1 1 0]]
[0.03790385 0.01186112 0.00875    0.00926572 0.01082283 0.00304609
 0.01030752 0.00926158 0.01030111 0.01082432 0.01082004 0.01238092
 0.01030078 0.00874296 0.0092672  0.01186426 0.01081725 0.01186085
 0.01030287 0.01134348 0.00874651 0.01186224 0.01081707 0.00978403
 0.01133692 0.01081925 0.01081828 0.00978281 0.01134357 0.01081925
 0.00823453 0.01134078 0.01186014 0.00977859 0.0087469  0.01238092
 0.01134603 0.006162   0.00978471 0.01186224 0.00875282 0.01134105
 0.03534282 0.03459665 0.03416143 0.0335284  0.03278252 0.0321809
 0.03160657 0.03110331 0.03014858 0.02972648 0.02921036 0.02874147
 0.02809787 0.02745904 0.02692057 0.02574315 0.02573228 0.02515223]


In [ ]:
import numpy as np
def page_rank(matrix, d=0.85, tol=1e-6, max_iter=10000):
    
    out_degree = matrix.sum(axis=1)
    
    weight = matrix / out_degree
    print(weight)
    N = matrix.shape[0]
    pr = np.ones(N).reshape(N, ) * 1.0 / N

    for iter in range(max_iter):
        old_pr = pr.copy()
        pr = d * weight.dot(pr) + (1 - d)/ N
        # yield nodes, pr, it
        err = np.absolute(pr - old_pr).sum()
        if err < tol:
            return pr.T
    return pr.T
    
if __name__ == "__main__":
    res = page_rank(matrix)
    print(res)

[[0.         0.05555556 0.08333333 ... 0.02380952 0.02380952 0.02439024]
 [0.01694915 0.         0.         ... 0.02380952 0.02380952 0.02439024]
 [0.01694915 0.         0.         ... 0.         0.         0.        ]
 ...
 [0.01694915 0.05555556 0.         ... 0.         0.         0.02439024]
 [0.01694915 0.05555556 0.         ... 0.         0.         0.02439024]
 [0.01694915 0.05555556 0.         ... 0.02380952 0.02380952 0.        ]]
[0.03790429 0.01186095 0.00874988 0.0092656  0.01082268 0.00304608
 0.01030738 0.00926146 0.01030097 0.01082417 0.01081989 0.01238075
 0.01030064 0.00874284 0.00926708 0.0118641  0.0108171  0.01186069
 0.01030273 0.01134332 0.00874639 0.01186207 0.01081692 0.00978389
 0.01133676 0.01081909 0.01081813 0.00978268 0.01134341 0.01081909
 0.00823443 0.01134062 0.01185997 0.00977846 0.00874678 0.01238075
 0.01134587 0.00616192 0.00978457 0.01186207 0.00875271 0.01134089
 0.03534324 0.03459705 0.03416182 0.03352877 0.03278288 0.03218124
 0.0316069  0.031103